# components/stats_panel

> Stats summary and verify selection button

In [ ]:
#| default_exp components.stats_panel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Optional

from fasthtml.common import Div, Span, Button, P

from cjm_fasthtml_daisyui.components.actions.button import btn, btn_sizes, btn_colors, btn_styles
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, text_align
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap, flex_wrap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

from cjm_transcription_source_select.models import SourceSelectUrls, SelectedFile, ExtractionResult
from cjm_transcription_source_select.html_ids import SourceSelectHtmlIds

## Stats Panel

Shows file counts and the verify selection button. After verification, shows a success message.

In [ ]:
#| export
def render_stats_panel(
    selected_files: List[SelectedFile],  # Current selection
    urls: SourceSelectUrls,  # URL bundle
    extraction_results: Optional[Dict[str, ExtractionResult]] = None,  # Extraction results
    verified: bool = False,  # Whether selection is verified
) -> Div:  # Stats panel component
    """Render stats summary with verify button."""
    extraction_results = extraction_results or {}
    audio_count = sum(1 for f in selected_files if f.get("file_type") == "audio")
    video_count = sum(1 for f in selected_files if f.get("file_type") == "video")
    total = len(selected_files)

    # Count extraction statuses
    videos_needing_extraction = 0
    videos_extracted = 0
    videos_with_error = 0
    for f in selected_files:
        if f.get("file_type") == "video":
            result = extraction_results.get(f["path"])
            if result and result.get("status") == "complete":
                videos_extracted += 1
            elif result and result.get("status") == "error":
                videos_with_error += 1
            else:
                videos_needing_extraction += 1

    # Empty state
    if total == 0:
        return Div(
            P(
                "Select audio or video files to begin",
                cls=combine_classes(text_dui.base_content.opacity(40), font_size.sm, text_align.center)
            ),
            id=SourceSelectHtmlIds.STATS_PANEL,
            cls=combine_classes(
                w.full, p(4), m.t(4),
                bg_dui.base_100, border_radius.box,
                shadow.lg, border_dui.base_300, border()
            )
        )

    # Stats text
    parts = [f"{total} file{'s' if total != 1 else ''} selected"]
    if audio_count:
        parts.append(f"{audio_count} audio")
    if video_count:
        video_detail = f"{video_count} video"
        if videos_needing_extraction > 0 and not verified:
            video_detail += f" (needs extraction)"
        elif videos_extracted == video_count:
            video_detail += " (extracted)"
        parts.append(video_detail)
    stats_text = " \u00b7 ".join(parts)

    # Verified state
    if verified:
        return Div(
            Div(
                Div(
                    lucide_icon("circle-check", size=5, cls=str(text_dui.success)),
                    Span(
                        f"Selection verified \u00b7 {total} audio source{'s' if total != 1 else ''} ready",
                        cls=combine_classes(font_size.sm, text_dui.success, m.l(2))
                    ),
                    cls=combine_classes(str(flex_display), items.center)
                ),
                cls=combine_classes(
                    str(flex_display), justify.center, items.center, p(4)
                )
            ),
            id=SourceSelectHtmlIds.STATS_PANEL,
            cls=combine_classes(
                w.full, m.t(4),
                bg_dui.base_100, border_radius.box,
                shadow.lg, border_dui.base_300, border()
            )
        )

    # Unverified state with button
    return Div(
        Div(
            Span(stats_text, cls=combine_classes(font_size.sm, text_dui.base_content)),
            Button(
                "Verify Selection",
                cls=combine_classes(btn, btn_colors.primary, btn_sizes.sm),
                hx_post=urls.verify,
                hx_target=SourceSelectHtmlIds.as_selector(SourceSelectHtmlIds.STATS_PANEL),
                hx_swap="outerHTML",
            ),
            cls=combine_classes(
                str(flex_display), justify.between, items.center,
                flex_wrap.wrap, gap(4), p(4)
            )
        ),
        id=SourceSelectHtmlIds.STATS_PANEL,
        cls=combine_classes(
            w.full, m.t(4),
            bg_dui.base_100, border_radius.box,
            shadow.lg, border_dui.base_300, border()
        )
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()